# 🍳 GPT-2 Recipe Pre-training from Scratch

**Build and pre-train a GPT-2 Mini language model (~50M parameters) on a custom recipe dataset**

---

## 📋 Overview

This notebook trains a GPT-2 language model from scratch on a recipe corpus to generate coherent recipe text.

**Pipeline:**
1. Train custom BPE tokenizer (30K vocabulary)
2. Initialize GPT-2 Mini architecture (6 layers, 512 dim, 8 heads)
3. Pre-train with FP16 mixed precision for 10 epochs
4. Generate recipe text from prompts

---

## ⚙️ Setup Instructions (Google Colab Pro)

1. **Enable A100 GPU**: `Runtime` → `Change runtime type` → `A100 GPU`
2. **Upload your recipe file**: Click folder icon → Upload `recipes.txt`
3. **Update `RECIPE_FILE_PATH`** in Section 0.0 below
4. **Run all cells**: `Runtime` → `Run all`

**Estimated Time**: ~2 hours for complete training on 8,500 recipes

---

## 📁 Outputs

- `outputs/tokenizer/` — Trained BPE tokenizer (vocab.json, merges.txt)
- `outputs/gpt2-recipe-checkpoints/` — Model checkpoints (epoch 1-10)

---

# SECTION 0: USER INPUTS & HYPERPARAMETERS

> ⚠️ **CONFIGURE THESE BEFORE RUNNING THE NOTEBOOK**

In [ ]:
# ============================================================================
# SECTION 0.0: USER INPUTS (MODIFY THESE BEFORE RUNNING)
# ============================================================================
"""
📁 Path to your recipe dataset file.
   Upload to Colab runtime first, then set the path here.
   Format: Plain text file, one recipe per line with [BOS]/[EOS] markers.
"""

RECIPE_FILE_PATH = "recipes.txt"  # ⚠️ INPUT REQUIRED: Set your file path here

In [ ]:
# ============================================================================
# SECTION 0.1: TOKENIZER HYPERPARAMETERS
# ============================================================================
"""
Configuration for training the Byte Pair Encoding (BPE) tokenizer.
These settings determine vocabulary size and special token handling.
"""

TOKENIZER_CONFIG = {
    "vocab_size": 30_000,           # Target vocabulary size for BPE
    "min_frequency": 2,             # Minimum token frequency to include
    "special_tokens": [
        "[PAD]",                    # Padding token (ID: 0)
        "[UNK]",                    # Unknown token (ID: 1)
        "[BOS]",                    # Beginning of sequence (ID: 2)
        "[EOS]",                    # End of sequence (ID: 3)
    ],
}

print("✓ Tokenizer config loaded")
print(f"  Vocabulary size: {TOKENIZER_CONFIG['vocab_size']:,}")
print(f"  Special tokens: {TOKENIZER_CONFIG['special_tokens']}")

In [ ]:
# ============================================================================
# SECTION 0.2: MODEL ARCHITECTURE HYPERPARAMETERS (GPT-2 Mini)
# ============================================================================
"""
GPT-2 Mini configuration: ~50M parameters.
Optimized for training from scratch on domain-specific data.
"""

MODEL_CONFIG = {
    "vocab_size": 30_000,           # Must match tokenizer vocab_size
    "n_positions": 3000,            # Maximum sequence length (context window)
    "n_embd": 512,                  # Embedding dimension
    "n_layer": 6,                   # Number of transformer layers
    "n_head": 8,                    # Number of attention heads (must divide n_embd)
    "activation_function": "gelu_new",
    "resid_pdrop": 0.1,             # Residual dropout
    "embd_pdrop": 0.1,              # Embedding dropout
    "attn_pdrop": 0.1,              # Attention dropout
}

# Calculate approximate parameter count
approx_params = (
    MODEL_CONFIG["vocab_size"] * MODEL_CONFIG["n_embd"] +  # Token embeddings
    MODEL_CONFIG["n_positions"] * MODEL_CONFIG["n_embd"] +  # Position embeddings
    MODEL_CONFIG["n_layer"] * (
        4 * MODEL_CONFIG["n_embd"] ** 2 +  # Attention (Q, K, V, O)
        8 * MODEL_CONFIG["n_embd"] ** 2    # FFN (up + down projection)
    )
)

print("✓ Model config loaded (GPT-2 Mini)")
print(f"  Layers: {MODEL_CONFIG['n_layer']}, Embedding: {MODEL_CONFIG['n_embd']}, Heads: {MODEL_CONFIG['n_head']}")
print(f"  Max sequence length: {MODEL_CONFIG['n_positions']:,} tokens")
print(f"  Approximate parameters: ~{approx_params / 1e6:.1f}M")

In [ ]:
# ============================================================================
# SECTION 0.3: TRAINING HYPERPARAMETERS
# ============================================================================
"""
Training configuration optimized for Google Colab Pro A100 GPU (40GB VRAM).
Effective batch size = per_device_train_batch_size × gradient_accumulation_steps = 16
"""

TRAINING_CONFIG = {
    "num_train_epochs": 10,                    # Total training epochs
    "per_device_train_batch_size": 4,          # Batch size per GPU (A100 40GB allows larger batches)
    "gradient_accumulation_steps": 4,          # Effective batch size = 4 * 4 = 16
    "learning_rate": 5e-5,                     # Peak learning rate
    "weight_decay": 0.01,                      # L2 regularization
    "warmup_steps": 500,                       # Linear warmup steps
    "fp16": True,                              # Mixed precision training
    "logging_dir": "./logs",                   # TensorBoard logs directory
    "logging_steps": 100,                      # Log every N steps
    "save_strategy": "epoch",                  # Save checkpoint every epoch
    "save_total_limit": 10,                    # Keep all 10 epoch checkpoints
    "output_dir": "./gpt2-recipe-checkpoints", # Checkpoint directory
    "report_to": "none",                       # Disable wandb/tensorboard
    "dataloader_num_workers": 2,               # Data loading workers
    "seed": 42,                                # Random seed for reproducibility
}

print("✓ Training config loaded")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Batch size: {TRAINING_CONFIG['per_device_train_batch_size']} × {TRAINING_CONFIG['gradient_accumulation_steps']} = {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']} effective")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  FP16: {TRAINING_CONFIG['fp16']}")

In [ ]:
# ============================================================================
# SECTION 0.4: INFERENCE HYPERPARAMETERS
# ============================================================================
"""
Generation settings for recipe text inference.
These control the diversity and quality of generated text.
"""

INFERENCE_CONFIG = {
    "max_new_tokens": 200,          # Maximum tokens to generate
    "temperature": 0.8,             # Sampling temperature (higher = more random)
    "top_k": 50,                    # Top-k sampling
    "top_p": 0.92,                  # Nucleus sampling threshold
    "do_sample": True,              # Enable sampling (vs greedy)
    "repetition_penalty": 1.1,      # Penalize repeated tokens
    "no_repeat_ngram_size": 3,      # Prevent repeating n-grams of this size
}

print("✓ Inference config loaded")
print(f"  Max new tokens: {INFERENCE_CONFIG['max_new_tokens']}")
print(f"  Temperature: {INFERENCE_CONFIG['temperature']}")
print(f"  Sampling: top_k={INFERENCE_CONFIG['top_k']}, top_p={INFERENCE_CONFIG['top_p']}")

---

# SECTION 1: ENVIRONMENT SETUP

> Install dependencies and verify GPU availability

In [ ]:
# ============================================================================
# SECTION 1.1: INSTALL DEPENDENCIES
# ============================================================================
"""
Install required packages. PyTorch is pre-installed in Colab.
Using -q for quiet installation to reduce output noise.
"""

!pip install -q transformers tokenizers datasets matplotlib seaborn

print("✓ Dependencies installed")

In [ ]:
# ============================================================================
# SECTION 1.2: IMPORT LIBRARIES
# ============================================================================
"""
Import all required libraries for tokenizer training, model building, and visualization.
"""

import os
import json
from pathlib import Path

# PyTorch (pre-installed in Colab)
import torch
from torch.utils.data import Dataset, DataLoader

# Hugging Face ecosystem
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Visualization (constitution-mandated)
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✓ All libraries imported successfully")
print(f"  PyTorch version: {torch.__version__}")
print(f"  Transformers imported")

In [ ]:
# ============================================================================
# SECTION 1.3: GPU AVAILABILITY CHECK
# ============================================================================
"""
Verify GPU availability and print device information.
This notebook is optimized for A100 GPU (40GB VRAM).
"""

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print("✓ GPU Available!")
    print(f"  Device: {gpu_name}")
    print(f"  VRAM: {gpu_memory:.1f} GB")
    
    # Check if A100
    if "A100" in gpu_name:
        print("  ✓ A100 GPU detected - optimal configuration")
    else:
        print(f"  ⚠️ Warning: Expected A100, got {gpu_name}")
        print("    Training may be slower or require batch size adjustment")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected - training will be very slow")
    print("  Recommendation: Enable GPU in Runtime > Change runtime type > A100")

print(f"\n  Using device: {device}")

---

# SECTION 2: DATA LOADING

> Load and explore the recipe dataset

In [ ]:
# ============================================================================
# SECTION 2.1: LOAD RECIPE TEXT FILE
# ============================================================================
"""
Load the recipe dataset from a plain text file.
Expected format: One recipe per line with [BOS]/[EOS] markers.
"""

# Check if file exists
if not os.path.exists(RECIPE_FILE_PATH):
    raise FileNotFoundError(
        f"Recipe file not found: {RECIPE_FILE_PATH}\n"
        "Please upload your recipe file to the Colab runtime and update RECIPE_FILE_PATH."
    )

# Load recipes
with open(RECIPE_FILE_PATH, "r", encoding="utf-8") as f:
    recipes = [line.strip() for line in f if line.strip()]

print(f"✓ Loaded {len(recipes):,} recipes from {RECIPE_FILE_PATH}")
print(f"\n📄 Sample recipe (first entry):")
print("-" * 60)
print(recipes[0][:500] + "..." if len(recipes[0]) > 500 else recipes[0])
print("-" * 60)

In [ ]:
# ============================================================================
# SECTION 2.2: DATA EXPLORATION & STATISTICS
# ============================================================================
"""
Compute and display statistics about the recipe dataset.
"""

import numpy as np

# Calculate recipe lengths (in characters)
recipe_lengths = [len(recipe) for recipe in recipes]

print("📊 Dataset Statistics")
print("=" * 40)
print(f"  Total recipes: {len(recipes):,}")
print(f"  Min length: {min(recipe_lengths):,} chars")
print(f"  Max length: {max(recipe_lengths):,} chars")
print(f"  Mean length: {np.mean(recipe_lengths):,.1f} chars")
print(f"  Median length: {np.median(recipe_lengths):,.1f} chars")
print(f"  Std deviation: {np.std(recipe_lengths):,.1f} chars")

# Check for [BOS] and [EOS] markers
bos_count = sum(1 for r in recipes if "[BOS]" in r)
eos_count = sum(1 for r in recipes if "[EOS]" in r)
print(f"\n📌 Special Token Coverage")
print(f"  Recipes with [BOS]: {bos_count:,} ({100*bos_count/len(recipes):.1f}%)")
print(f"  Recipes with [EOS]: {eos_count:,} ({100*eos_count/len(recipes):.1f}%)")

In [ ]:
# ============================================================================
# SECTION 2.3: VISUALIZE RECIPE LENGTH DISTRIBUTION
# ============================================================================
"""
Plot the distribution of recipe lengths using seaborn histogram.
"""

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(recipe_lengths, bins=50, kde=True, ax=ax, color="steelblue")

ax.set_xlabel("Recipe Length (characters)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of Recipe Lengths in Dataset", fontsize=14)

# Add vertical line for max sequence length (in chars, approximating 3000 tokens)
# Rough estimate: 1 token ≈ 4 characters
approx_max_chars = MODEL_CONFIG["n_positions"] * 4
ax.axvline(x=approx_max_chars, color="red", linestyle="--", linewidth=2, 
           label=f"Max seq length (~{approx_max_chars:,} chars)")
ax.legend()

plt.tight_layout()
plt.show()

# Count recipes that will be truncated
truncated_count = sum(1 for l in recipe_lengths if l > approx_max_chars)
print(f"\n⚠️ Recipes exceeding max length: {truncated_count:,} ({100*truncated_count/len(recipes):.1f}%)")
print("   These will be truncated during tokenization.")

---

# SECTION 3: TOKENIZER TRAINING

> Train a custom Byte Pair Encoding (BPE) tokenizer on the recipe corpus

In [ ]:
# ============================================================================
# SECTION 3.1: TRAIN BPE TOKENIZER
# ============================================================================
# Train a Byte Pair Encoding tokenizer from scratch on recipe corpus

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors

# Initialize BPE tokenizer
tokenizer = Tokenizer(models.BPE(unk_token=TOKENIZER_CONFIG["special_tokens"][1]))

# Set pre-tokenizer (whitespace-based splitting)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Configure BPE trainer
trainer = trainers.BpeTrainer(
    vocab_size=TOKENIZER_CONFIG["vocab_size"],
    min_frequency=TOKENIZER_CONFIG["min_frequency"],
    special_tokens=TOKENIZER_CONFIG["special_tokens"],
    show_progress=True
)

# Train on recipe corpus
print("Training BPE tokenizer on recipe corpus...")
tokenizer.train_from_iterator(recipes, trainer=trainer)

# Add post-processor for BOS/EOS tokens
bos_token = TOKENIZER_CONFIG["special_tokens"][2]  # [BOS]
eos_token = TOKENIZER_CONFIG["special_tokens"][3]  # [EOS]
bos_id = tokenizer.token_to_id(bos_token)
eos_id = tokenizer.token_to_id(eos_token)

tokenizer.post_processor = processors.TemplateProcessing(
    single=f"{bos_token}:0 $A:0 {eos_token}:0",
    special_tokens=[
        (bos_token, bos_id),
        (eos_token, eos_id),
    ],
)

print(f"✓ Tokenizer trained successfully!")
print(f"  Vocabulary size: {tokenizer.get_vocab_size():,}")

In [ ]:
# ============================================================================
# SECTION 3.2: SAVE TOKENIZER TO DISK
# ============================================================================
# Persist trained tokenizer for checkpoint recovery

import os

TOKENIZER_PATH = "recipe_tokenizer"
os.makedirs(TOKENIZER_PATH, exist_ok=True)

# Save the raw tokenizer
tokenizer.save(os.path.join(TOKENIZER_PATH, "tokenizer.json"))

print(f"✓ Tokenizer saved to: {TOKENIZER_PATH}/")

In [ ]:
# ============================================================================
# SECTION 3.3: WRAP IN GPT2TokenizerFast
# ============================================================================
# Create HuggingFace-compatible tokenizer wrapper for Trainer API

from transformers import GPT2TokenizerFast

# Load trained tokenizer into HuggingFace wrapper
hf_tokenizer = GPT2TokenizerFast(
    tokenizer_file=os.path.join(TOKENIZER_PATH, "tokenizer.json"),
    bos_token=TOKENIZER_CONFIG["special_tokens"][2],
    eos_token=TOKENIZER_CONFIG["special_tokens"][3],
    unk_token=TOKENIZER_CONFIG["special_tokens"][1],
    pad_token=TOKENIZER_CONFIG["special_tokens"][0],
)

# Set padding side for causal LM (left padding)
hf_tokenizer.padding_side = "left"

print(f"✓ GPT2TokenizerFast wrapper created!")
print(f"  PAD token: {hf_tokenizer.pad_token} (id: {hf_tokenizer.pad_token_id})")
print(f"  UNK token: {hf_tokenizer.unk_token} (id: {hf_tokenizer.unk_token_id})")
print(f"  BOS token: {hf_tokenizer.bos_token} (id: {hf_tokenizer.bos_token_id})")
print(f"  EOS token: {hf_tokenizer.eos_token} (id: {hf_tokenizer.eos_token_id})")

In [ ]:
# ============================================================================
# SECTION 3.4: TOKENIZER VALIDATION DEMO
# ============================================================================
# Demonstrate tokenization on sample recipes

print("=" * 60)
print("TOKENIZER VALIDATION DEMO")
print("=" * 60)

# Sample recipes for demo
sample_recipes = recipes[:3]

for i, recipe in enumerate(sample_recipes):
    print(f"\n--- Recipe {i+1} ---")
    print(f"Original (first 100 chars): {recipe[:100]}...")
    
    # Tokenize
    encoded = hf_tokenizer(recipe, truncation=True, max_length=MODEL_CONFIG["n_positions"])
    tokens = hf_tokenizer.convert_ids_to_tokens(encoded["input_ids"][:20])
    
    print(f"Token count: {len(encoded['input_ids'])}")
    print(f"First 20 tokens: {tokens}")

print("\n" + "=" * 60)
print("✓ Tokenizer validation complete!")

---

# SECTION 4: DATASET PREPARATION

> Create PyTorch Dataset and DataCollator for training

In [ ]:
# ============================================================================
# SECTION 4.1: RECIPE DATASET CLASS
# ============================================================================
# Custom PyTorch Dataset for tokenized recipes

from torch.utils.data import Dataset

class RecipeDataset(Dataset):
    """PyTorch Dataset for recipe text generation training."""
    
    def __init__(self, recipes: list, tokenizer, max_length: int):
        """
        Args:
            recipes: List of recipe strings
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
        """
        self.recipes = recipes
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.recipes)
    
    def __getitem__(self, idx):
        recipe = self.recipes[idx]
        
        # Tokenize with truncation
        encoding = self.tokenizer(
            recipe,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"]
        }

print("✓ RecipeDataset class defined!")

In [ ]:
# ============================================================================
# SECTION 4.2: TOKENIZE ALL RECIPES
# ============================================================================
# Create training dataset from recipe corpus

# Create dataset
train_dataset = RecipeDataset(
    recipes=recipes,
    tokenizer=hf_tokenizer,
    max_length=MODEL_CONFIG["n_positions"]
)

print(f"✓ Training dataset created!")
print(f"  Total samples: {len(train_dataset):,}")
print(f"  Max sequence length: {MODEL_CONFIG['n_positions']:,}")

# Sample verification
sample = train_dataset[0]
print(f"\n  Sample 0 token count: {len(sample['input_ids'])}")

In [ ]:
# ============================================================================
# SECTION 4.3: DATA COLLATOR FOR CAUSAL LM
# ============================================================================
# Configure DataCollator for language modeling (shifts labels automatically)

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=False,  # Causal LM (not masked LM)
)

print("✓ DataCollatorForLanguageModeling configured!")
print("  Mode: Causal Language Modeling (mlm=False)")
print("  Labels are automatically shifted for next-token prediction")

---

# SECTION 5: MODEL INITIALIZATION

> Configure and instantiate GPT-2 Mini architecture

In [ ]:
# ============================================================================
# SECTION 5.1: CREATE GPT2CONFIG
# ============================================================================
# Configure GPT-2 Mini architecture

from transformers import GPT2Config

config = GPT2Config(
    vocab_size=MODEL_CONFIG["vocab_size"],
    n_positions=MODEL_CONFIG["n_positions"],
    n_embd=MODEL_CONFIG["n_embd"],
    n_layer=MODEL_CONFIG["n_layer"],
    n_head=MODEL_CONFIG["n_head"],
    # n_inner defaults to 4 * n_embd = 2048
    activation_function=MODEL_CONFIG["activation_function"],
    resid_pdrop=MODEL_CONFIG["resid_pdrop"],
    embd_pdrop=MODEL_CONFIG["embd_pdrop"],
    attn_pdrop=MODEL_CONFIG["attn_pdrop"],
    bos_token_id=hf_tokenizer.bos_token_id,
    eos_token_id=hf_tokenizer.eos_token_id,
    pad_token_id=hf_tokenizer.pad_token_id,
)

print("✓ GPT2Config created!")
print(f"  Architecture: GPT-2 Mini")
print(f"  Vocab size: {config.vocab_size:,}")
print(f"  Max positions: {config.n_positions:,}")
print(f"  Embedding dim: {config.n_embd}")
print(f"  Layers: {config.n_layer}")
print(f"  Heads: {config.n_head}")
print(f"  FFN inner dim: {config.n_inner} (default: 4 * n_embd)")

In [ ]:
# ============================================================================
# SECTION 5.2: INSTANTIATE GPT2LMHeadModel
# ============================================================================
# Create model from scratch (random initialization)

from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel(config)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✓ GPT2LMHeadModel instantiated!")
print(f"  Device: {device}")

In [ ]:
# ============================================================================
# SECTION 5.3: MODEL PARAMETER COUNT
# ============================================================================
# Display total trainable parameters

def count_parameters(model):
    """Count trainable and total parameters."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable_params, total_params = count_parameters(model)

print("=" * 60)
print("MODEL PARAMETER SUMMARY")
print("=" * 60)
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Trainable parameters: {trainable_params:>15,}")
print(f"  Approximate size:     {total_params * 4 / 1e6:>12.2f} MB (FP32)")
print(f"  Approximate size:     {total_params * 2 / 1e6:>12.2f} MB (FP16)")
print("=" * 60)

---

# SECTION 6: TRAINING

> Configure TrainingArguments, initialize Trainer, and execute training loop

In [ ]:
# ============================================================================
# SECTION 6.1: TRAINING ARGUMENTS
# ============================================================================
# Configure HuggingFace Trainer hyperparameters

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=TRAINING_CONFIG["output_dir"],
    overwrite_output_dir=True,
    
    # Training duration
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    
    # Batch size and accumulation
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # Optimizer settings
    learning_rate=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    warmup_steps=TRAINING_CONFIG["warmup_steps"],
    
    # Mixed precision
    fp16=TRAINING_CONFIG["fp16"],
    
    # Logging
    logging_dir=TRAINING_CONFIG["logging_dir"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    
    # Checkpointing
    save_strategy=TRAINING_CONFIG["save_strategy"],
    save_total_limit=TRAINING_CONFIG["save_total_limit"],
    
    # Misc
    dataloader_num_workers=TRAINING_CONFIG["dataloader_num_workers"],
    seed=TRAINING_CONFIG["seed"],
    report_to=TRAINING_CONFIG["report_to"],
)

print("✓ TrainingArguments configured!")
print(f"  Output directory: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# ============================================================================
# SECTION 6.2: INITIALIZE TRAINER
# ============================================================================
# Create HuggingFace Trainer instance

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=hf_tokenizer,
)

print("✓ Trainer initialized!")
print(f"  Model: {model.__class__.__name__}")
print(f"  Dataset size: {len(train_dataset):,} samples")

In [ ]:
# ============================================================================
# SECTION 6.3: EXECUTE TRAINING
# ============================================================================
# Run the training loop

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"  Model: GPT-2 Mini ({total_params:,} parameters)")
print(f"  Dataset: {len(train_dataset):,} recipes")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Device: {device}")
print("=" * 60)

# Train the model
train_result = trainer.train()

# Print training summary
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Total steps: {train_result.global_step:,}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print("=" * 60)

In [ ]:
# ============================================================================
# SECTION 6.4: PLOT TRAINING LOSS CURVE
# ============================================================================
# Visualize training progress

# Extract loss history from trainer
loss_history = [log["loss"] for log in trainer.state.log_history if "loss" in log]
steps = [log["step"] for log in trainer.state.log_history if "loss" in log]

# Create loss curve plot
plt.figure(figsize=(12, 6))
plt.plot(steps, loss_history, 'b-', linewidth=2, alpha=0.8)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("GPT-2 Recipe Pre-training Loss Curve", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3)

# Add epoch markers
samples_per_epoch = len(train_dataset)
steps_per_epoch = samples_per_epoch // (TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"])
for epoch in range(1, TRAINING_CONFIG["num_train_epochs"] + 1):
    epoch_step = epoch * steps_per_epoch
    if epoch_step <= max(steps):
        plt.axvline(x=epoch_step, color='r', linestyle='--', alpha=0.5, label=f'Epoch {epoch}' if epoch == 1 else '')

plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Training loss curve saved to: training_loss_curve.png")

---

# SECTION 7: INFERENCE

> Generate recipe text using the trained model

In [ ]:
# ============================================================================
# SECTION 7.1: LOAD BEST CHECKPOINT
# ============================================================================
# Load the best checkpoint for inference

import glob

# Find the latest checkpoint
checkpoint_dirs = glob.glob(os.path.join(TRAINING_CONFIG["output_dir"], "checkpoint-*"))
if checkpoint_dirs:
    latest_checkpoint = max(checkpoint_dirs, key=os.path.getctime)
    print(f"✓ Loading checkpoint: {latest_checkpoint}")
    model = GPT2LMHeadModel.from_pretrained(latest_checkpoint)
    model = model.to(device)
else:
    print("ℹ Using current model (no checkpoint found)")

model.eval()
print(f"✓ Model ready for inference on {device}")

In [ ]:
# ============================================================================
# SECTION 7.2: RECIPE GENERATION FUNCTION
# ============================================================================
# Define text generation function with configurable parameters

def generate_recipe(prompt: str, max_new_tokens: int = None, temperature: float = None,
                   top_k: int = None, top_p: float = None, do_sample: bool = True) -> str:
    """
    Generate recipe text from a prompt.
    
    Args:
        prompt: Starting text for generation
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (higher = more creative)
        top_k: Top-k sampling parameter
        top_p: Nucleus sampling parameter
        do_sample: Whether to use sampling (False = greedy)
    
    Returns:
        Generated recipe text
    """
    # Use defaults from config if not specified
    max_new_tokens = max_new_tokens or INFERENCE_CONFIG["max_new_tokens"]
    temperature = temperature or INFERENCE_CONFIG["temperature"]
    top_k = top_k or INFERENCE_CONFIG["top_k"]
    top_p = top_p or INFERENCE_CONFIG["top_p"]
    
    # Tokenize prompt
    inputs = hf_tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=hf_tokenizer.pad_token_id,
            eos_token_id=hf_tokenizer.eos_token_id,
            repetition_penalty=INFERENCE_CONFIG["repetition_penalty"],
            no_repeat_ngram_size=INFERENCE_CONFIG["no_repeat_ngram_size"],
        )
    
    # Decode and return
    generated_text = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

print("✓ generate_recipe() function defined!")

In [ ]:
# ============================================================================
# SECTION 7.3: INTERACTIVE GENERATION EXAMPLES
# ============================================================================
# Demonstrate recipe generation with various prompts

print("=" * 60)
print("RECIPE GENERATION EXAMPLES")
print("=" * 60)

# Example prompts
prompts = [
    "Chocolate Cake:",
    "Pasta with",
    "Grilled Chicken",
    "Easy breakfast",
]

for prompt in prompts:
    print(f"\n{'─' * 60}")
    print(f"PROMPT: {prompt}")
    print('─' * 60)
    
    generated = generate_recipe(prompt)
    print(generated[:500] + "..." if len(generated) > 500 else generated)

print("\n" + "=" * 60)
print("✓ Generation examples complete!")
print("=" * 60)

---

# SECTION 8: EXPORT & CLEANUP

> Save final model artifacts and prepare for download

In [ ]:
# ============================================================================
# SECTION 8.1: SAVE FINAL MODEL
# ============================================================================
# Save model and tokenizer for future use

FINAL_MODEL_PATH = "gpt2_recipe_final"

# Save model
model.save_pretrained(FINAL_MODEL_PATH)
print(f"✓ Model saved to: {FINAL_MODEL_PATH}/")

# Save tokenizer
hf_tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"✓ Tokenizer saved to: {FINAL_MODEL_PATH}/")

# List saved files
saved_files = os.listdir(FINAL_MODEL_PATH)
print(f"\nSaved artifacts:")
for f in saved_files:
    size = os.path.getsize(os.path.join(FINAL_MODEL_PATH, f)) / 1e6
    print(f"  {f}: {size:.2f} MB")

In [ ]:
# ============================================================================
# SECTION 8.2: DOWNLOAD ARTIFACTS (COLAB)
# ============================================================================
# Zip and download model artifacts for local storage

import shutil

# Create zip archive
zip_name = "gpt2_recipe_model"
shutil.make_archive(zip_name, 'zip', FINAL_MODEL_PATH)
print(f"✓ Created archive: {zip_name}.zip")

# Download in Colab (uncomment when running in Colab)
# from google.colab import files
# files.download(f"{zip_name}.zip")

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print(f"  Model: GPT-2 Mini (~{total_params:,} parameters)")
print(f"  Dataset: {len(train_dataset):,} recipes")
print(f"  Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  Final Loss: {train_result.training_loss:.4f}")
print("=" * 60)
print("\nTo download in Colab, uncomment the files.download() line above.")
print("To load the model later:")
print(f"  model = GPT2LMHeadModel.from_pretrained('{FINAL_MODEL_PATH}')")
print(f"  tokenizer = GPT2TokenizerFast.from_pretrained('{FINAL_MODEL_PATH}')")